In [19]:
import pandas as pd
from sqlalchemy import create_engine
import urllib
from datetime import datetime
import os
from dotenv import load_dotenv

load_dotenv()

True

In [20]:
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={os.getenv('DB_SERVER')};"
    "DATABASE=Ventas_Comerssia;"
    f"UID={os.getenv('DB_USER')};"
    f"PWD={os.getenv('DB_PASSWORD')};"
)

# Crear el motor de conexión
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [21]:
fecha_corte = pd.to_datetime("2026-03-31")

# Rango de análisis 
fecha_inicio_12m = fecha_corte - pd.DateOffset(months=12)
fecha_inicio_24m = fecha_corte - pd.DateOffset(months=24)

In [22]:
# ===============================
# Datos de clientes y ventas
# ===============================
df_clientes = pd.read_sql(f"""
SELECT 
    c.Cliente,
    m.ID_Cliente_Unico,
    c.CosechaFecha
FROM Ventas_Comerssia.dbo.Cliente_Perfil c
LEFT JOIN Ventas_Comerssia.dbo.MASTER_ID m
    ON c.Cliente = m.Cliente
WHERE c.CosechaFecha <= '{fecha_corte}'
""", engine)
df_clientes["CosechaFecha"] = pd.to_datetime(df_clientes["CosechaFecha"], errors="coerce")

query_ventas = f"""
SELECT 
    Cliente,
    ID_Cliente_Unico,
    Fecha,
    Venta_Neta AS Venta
FROM Ventas_Comerssia.dbo.Ventas_Unificadas
WHERE Fecha BETWEEN '{fecha_inicio_24m:%Y-%m-%d}' AND '{fecha_corte:%Y-%m-%d}'
"""

df_ventas = pd.read_sql(query_ventas, engine)
df_ventas["Fecha"] = pd.to_datetime(df_ventas["Fecha"], errors="coerce")

In [23]:
# ===============================
# Validacion de cosecha por ID
# ===============================
ids_totales = df_clientes["ID_Cliente_Unico"].nunique()
ids_sin_cosecha = df_clientes["CosechaFecha"].isna().sum()

print("\n📌 Validacion Cosecha por ID_Cliente_Unico")
print(f"  IDs unicos totales: {ids_totales:,}")
print(f"  IDs sin CosechaFecha: {ids_sin_cosecha:,} ({ids_sin_cosecha / max(len(df_clientes), 1) * 100:.2f}%)")

if ids_totales > 0:
    min_cosecha = df_clientes["CosechaFecha"].min()
    max_cosecha = df_clientes["CosechaFecha"].max()
    print(f"  Primera cosecha registrada: {min_cosecha}")
    print(f"  Ultima cosecha registrada:  {max_cosecha}")

print("\nTop 10 IDs sin CosechaFecha (si existen):")
print(df_clientes.loc[df_clientes["CosechaFecha"].isna(), ["ID_Cliente_Unico"]].head(10).to_string(index=False))


📌 Validacion Cosecha por ID_Cliente_Unico
  IDs unicos totales: 280,064
  IDs sin CosechaFecha: 0 (0.00%)
  Primera cosecha registrada: 2019-09-01 00:00:00
  Ultima cosecha registrada:  2026-03-31 00:00:00

Top 10 IDs sin CosechaFecha (si existen):
Empty DataFrame
Columns: [ID_Cliente_Unico]
Index: []


In [24]:
# ===============================
# 3) Métricas por cliente
# ===============================

# Para clientes con múltiples registros, se toma la primera fecha de cosecha (más antigua) - CONSOLIDAR CLIENTES (1 FILA POR ID)
df_clientes = (
    df_clientes
    .sort_values("CosechaFecha")
    .groupby("ID_Cliente_Unico", as_index=False)
    .first()
)

# Última compra 
ultima_compra = (
    df_ventas.groupby("ID_Cliente_Unico")["Fecha"]
    .max()
    .reset_index()
    .rename(columns={"Fecha": "UltimaCompra"})
)

# Ventas últimos 12 meses (para Segmento Actual)
ventas_12m = (
    df_ventas[df_ventas["Fecha"] >= fecha_inicio_12m]
    .groupby("ID_Cliente_Unico")["Venta"]
    .sum()
    .reset_index()
    .rename(columns={"Venta": "Venta12M"})
)

# Ventas 24 meses (para Segmento24M)
ventas_24m = (
    df_ventas.groupby("ID_Cliente_Unico")["Venta"]
    .sum()
    .reset_index()
    .rename(columns={"Venta": "Venta24M"})
)

# ===============================
# 4) Merge sobre la base completa de clientes
# ===============================
clientes = df_clientes.merge(ultima_compra, on="ID_Cliente_Unico", how="left")
clientes = clientes.merge(ventas_12m, on="ID_Cliente_Unico", how="left")
clientes = clientes.merge(ventas_24m, on="ID_Cliente_Unico", how="left")


In [25]:
# ===============================
# 5) Función exacta para asignar segmento por monto 
# ===============================
def segmento_por_valor(valor):
    if pd.isna(valor) or valor == 0:
        return "Sin Segmento"
    elif valor > 1_400_000:
        return "Diamante"
    elif valor >= 700_000:
        return "Oro"
    elif valor >= 300_000:
        return "Plata"
    else:
        return "Bronce"

clientes["Segmento12M"] = clientes["Venta12M"].apply(segmento_por_valor)
clientes["Segmento24M"] = clientes["Venta24M"].apply(segmento_por_valor)

# ===============================
# 6) Recencia en días 
# ===============================
# calcular dias desde ultima compra 
clientes["RecenciaDias"] = (fecha_corte - clientes["UltimaCompra"]).dt.days

def calcular_recencia(row):
    # Si no tiene ultima compra -> devolvemos 
    if pd.isna(row["UltimaCompra"]):
        return None

    # Nuevo: basado en cosecha (<= 90 dias)
    if pd.notna(row["CosechaFecha"]):
        diff_cosecha = (fecha_corte - row["CosechaFecha"]).days
        if diff_cosecha <= 90:
            return "Nuevo"

    d = row["RecenciaDias"]

    if pd.isna(d):
        return None
    if d <= 120:
        return "Muy Activo"
    elif d <= 300:
        return "Activo"
    elif d <= 365:
        return "Por Inactivar"
    elif d <= 395:
        return "Churn"
    else:
        return "Inactivo"

clientes["Recencia"] = clientes.apply(calcular_recencia, axis=1)

# ===============================
# 7) Definir Segmento final 
# ===============================
def asignar_segmento_final(row):
    rec = row["Recencia"]
    if rec in ["Nuevo", "Muy Activo", "Activo", "Por Inactivar"]:
        return row["Segmento12M"]
    if rec in ["Churn", "Inactivo"]:
        return row["Segmento24M"]
    # Si no tiene recencia (no compró) -> si tiene ventas24M ==0 -> Sin Segmento
    return "Sin Segmento"

clientes["Segmento"] = clientes.apply(asignar_segmento_final, axis=1)

# ===============================
# 8) Regla final: si Segmento 
# ===============================
clientes.loc[clientes["Segmento"] == "Sin Segmento", "Recencia"] = None

In [26]:
df_base_clientes = pd.read_sql("""
SELECT Cliente, ID_Cliente_Unico
FROM Ventas_Comerssia.dbo.MASTER_ID
""", engine)

resultado_final = df_base_clientes.merge(
    clientes[["ID_Cliente_Unico", "Segmento", "Recencia"]],
    on="ID_Cliente_Unico",
    how="left"
)

In [27]:
# ===============================
# 9) Resultado final (columns)
# ===============================
resultado = clientes[[
    "ID_Cliente_Unico", "Cliente","CosechaFecha", "UltimaCompra",
    "Venta12M", "Venta24M",
    "Recencia", "Segmento"
]]

# Mostrar ejemplo
print(resultado.head(50))

    ID_Cliente_Unico      Cliente CosechaFecha UltimaCompra    Venta12M  \
0           100001.0    C93398576   2019-09-29   2025-12-24   214285.72   
1           100004.0    C21335221   2020-11-21          NaT         NaN   
2           100007.0    C79126804   2021-07-10   2025-07-24   233613.44   
3           100009.0  C1044421468   2022-09-08          NaT         NaN   
4           100010.0    C30738398   2022-05-15   2025-11-01   390756.31   
5           100011.0  C1037591316   2019-12-20          NaT         NaN   
6           100015.0    C40398672   2020-02-26   2026-01-20   635714.29   
7           100016.0    C42165131   2020-11-01   2025-11-01   249579.83   
8           100017.0    C52409129   2022-03-15   2025-11-15   276890.76   
9           100018.0    C63342484   2021-12-17          NaT         NaN   
10          100019.0    C52992497   2019-11-26   2026-03-06  1206274.05   
11          100020.0    C88197904   2020-11-18          NaT         NaN   
12          100023.0    C

In [28]:
print("Clientes originales:", df_ventas["Cliente"].nunique())
print("Clientes reales:", df_ventas["ID_Cliente_Unico"].nunique())

Clientes originales: 134680
Clientes reales: 131321


In [29]:
resultado.to_excel("segmentacion_ID_UNICO.xlsx", index=False)

In [30]:
resultado.to_sql("Segmentacion_ID_UNICO", engine, if_exists="replace", index=False)

248